In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

#####1. Create your bronze layer tables ingesting from the landing zone

In [0]:
@dlt.table(
    name = "customers_raw_refactored",
    comment = "table to be created in bronze layer"
)
def get_customers_raw():
    df = (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter","*.csv")
        .option("header", "True")
        # schemaLocation REMOVED: DLT handles this automatically
        .option("cloudFiles.inferColumnTypes","True")
        .option("cloudFiles.schemaEvolutionMode", "none")
        .load("/Volumes/dbcertification/scholarsnest/landing_zone/customers/")
        .withColumn("load_time",current_timestamp())
        )
    return df

# .option("cloudFiles.schemaLocation", "/Volumes/dbcertification/scholarsnest/landing_zone/customers/schema/")
# .option("cloudFiles.schemaEvolutionMode", "addNewColumns")

In [0]:
@dlt.table(
    name = "invoices_raw_refactored",
    comment = "Raw invoices data optimized with Liquid Clustering.",
    # This replaces legacy partitioning:
    cluster_by = ["Country", "CustomerID"]
)
def get_invoices_raw():
    df = (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter","*.csv")
        .option("header", "True")
        .option("cloudFiles.inferColumnTypes","True")
        # Changed to 'rescue' to strictly adhere to the SLA while preventing data loss of new columns
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .load("/Volumes/dbcertification/scholarsnest/landing_zone/invoices/")
        .withColumn("load_time",current_timestamp())
        )
    return df
# .option("cloudFiles.schemaLocation", "/Volumes/dbcertification/scholarsnest/landing_zone/invoices/schema/")
# .option("cloudFiles.schemaEvolutionMode", "addNewColumns")

#####2. Create your silver layer tables reading incremental data from bronze layer

In [0]:
@dlt.table(
    name = "customers_cleaned_refactored",
    comment = "table to be created in silver layer"
)
@dlt.expect_or_drop("valid_customers","customer_id is not null")
def get_customers_cleaned():
    df = (
            # Replaced spark.readStream with dlt.read_stream for internal pipeline routing
            dlt.read_stream("LIVE.customers_raw_refactored")
            .selectExpr("CustomerID as customer_id","CustomerName as customer_name","load_time as load_time")
    )
    return df

In [0]:
@dlt.table(
    name = "invoices_cleaned_refactored",
    comment = "Cleaned invoice data optimized with Liquid Clustering.",
    cluster_by = ["invoice_year", "country"]
)
@dlt.expect_or_drop("valid_invoice_and_qty","invoice_no is not null and quantity > 0")
def get_invoices_cleaned():
    df = (
        # Replaced spark.readStream with dlt.read_stream
        dlt.read_stream("LIVE.invoices_raw_refactored")
          .selectExpr("InvoiceNo as invoice_no", 
                      "StockCode as stock_code", 
                      "Description as description",
                      "Quantity as quantity", 
                      "to_date(InvoiceDate, 'd-M-y H.m') as invoice_date", 
                      "UnitPrice as unit_price", 
                      "CustomerID as customer_id", 
                      "Country as country",
                      "year(to_date(InvoiceDate, 'd-M-y H.m')) as invoice_year", 
                      "month(to_date(InvoiceDate, 'd-M-y H.m')) as invoice_month", 
                      "load_time")
    )
    return df

#####3. Build your SCD Type 2 dimensions using CDC from silver layer

In [0]:
 dlt.create_streaming_table("customers_refactored") #The empty streaming table

dlt.apply_changes(
    target = "customers_refactored",
    source = "customers_cleaned_refactored",
    keys = ["customer_id"],
    sequence_by = col("load_time"), #for handling out-of-order data
    stored_as_scd_type = 2
)

#####4. Merge into your fact table using CDC from the silver layer

In [0]:
dlt.create_streaming_table("invoices_refactored",cluster_by = ["invoice_year","country"])

dlt.apply_changes(
    target = "invoices_refactored",
    source = "invoices_cleaned_refactored",
    keys = ["invoice_no", "stock_code", "invoice_date"],
    sequence_by = col("load_time")
)

#####5. Materialize your gold layer summary using silver layer fact

In [0]:
@dlt.table(
    name = "daily_sales_uk_2022_refactored"
)
def compute_daily_sales_uk_2022():
    return(
        dlt.read("LIVE.invoices_refactored")
        .where("invoice_year = 2022 AND country = 'United Kingdom'")
        .groupBy("country","invoice_year","invoice_month","invoice_date")
        .agg(expr("round(sum(quantity*unit_price),2)").alias("total_sales"))
    )

In [0]:
@dlt.table(
    name = "gold_enriched_invoices_refactored",
    comment = "Accurate fact table enriched with Point-in-Time customer dimension details."
)
def get_enriched_invoices():
    # 1. Read the materialized state of both tables
    facts_df = dlt.read("invoices_refactored")
    dims_df = dlt.read("customers_refactored")

    # 2. Perform the Point-in-Time Join
    # We join on customer_id AND ensure the invoice happened while that specific customer row was active.
    enriched_df = facts_df.join(
        dims_df,
        on = (
            (facts_df.customer_id == dims_df.customer_id) & 
            (facts_df.invoice_date >= dims_df.__START_AT) & 
            (
                (facts_df.invoice_date <= dims_df.__END_AT) | 
                (dims_df.__END_AT.isNull()) 
            )
        ),
        how = "left"
    )
    
    # 3. Select the final columns for your downstream BI tools
    # Dropping the duplicate customer_id and the DLT metadata columns
    return enriched_df.select(
        facts_df.invoice_no,
        facts_df.invoice_date,
        facts_df.stock_code,
        facts_df.quantity,
        facts_df.unit_price,
        facts_df.country.alias("invoice_country"),
        facts_df.customer_id.alias("invoices_customer_id"),
        dims_df.customer_id,
        dims_df.customer_name
    )